In [3]:
!pip install -q pymupdf faiss-gpu rank_bm25 sentence-transformers tqdm groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 82.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.9/134.9 MB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 15.5 MB/s eta 0:00:00


In [4]:
import fitz  # PyMuPDF
import numpy as np
import faiss
from tqdm.auto import tqdm
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder
from groq import Groq
from google.colab import userdata

# Initialize Models
# Use a small fast model for embeddings
embed_model = SentenceTransformer('all-MiniLM-L6-v2', device='cuda')
# Cross-encoder for reranking
rerank_model = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2', device='cuda')

# Client for Groq
client = Groq(api_key=userdata.get('GROQ_API_KEY'))

def extract_text_from_pdf(pdf_path):
    doc = fitz.open(pdf_path)
    chunks = []
    for page in tqdm(doc, desc="Extracting PDF"):
        text = page.get_text("text")
        # Simple chunking by paragraph/page
        if text.strip():
            chunks.append(text.strip())
    return chunks

pdf_path = '/content/Seerat e Mustafa_new.pdf'
doc_chunks = extract_text_from_pdf(pdf_path)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Extracting PDF:   0%|          | 0/675 [00:00<?, ?it/s]

In [5]:
# 1. Vector Indexing (Dense)
print("Encoding chunks for Vector Search...")
embeddings = embed_model.encode(doc_chunks, show_progress_bar=True)
dimension = embeddings.shape[1]
res = faiss.StandardGpuResources()
index = faiss.IndexFlatL2(dimension)
gpu_index = faiss.index_cpu_to_gpu(res, 0, index)
gpu_index.add(np.array(embeddings).astype('float32'))

# 2. BM25 Indexing (Sparse)
tokenized_corpus = [doc.split(" ") for doc in doc_chunks]
bm25 = BM25Okapi(tokenized_corpus)

Encoding chunks for Vector Search...


Batches:   0%|          | 0/21 [00:00<?, ?it/s]

In [11]:
def hybrid_retrieval(query, top_k=10):
    # Dense Search
    query_embedding = embed_model.encode([query])
    D, I = gpu_index.search(np.array(query_embedding).astype('float32'), top_k)
    vector_results = [doc_chunks[i] for i in I[0]]

    # Sparse Search
    tokenized_query = query.split(" ")
    bm25_results = bm25.get_top_n(tokenized_query, doc_chunks, n=top_k)

    # Combine (Union for simplicity in this demo)
    combined = list(set(vector_results + bm25_results))
    return combined

def rerank_results(query, documents):
    pairs = [[query, doc] for doc in documents]
    scores = rerank_model.predict(pairs)
    # Sort documents by score
    reranked = [doc for _, doc in sorted(zip(scores, documents), key=lambda x: x[0], reverse=True)]
    return reranked[:5]

# Example usage
query = "Describe the early life of the Prophet described in the document"
initial_docs = hybrid_retrieval(query)
final_docs = rerank_results(query, initial_docs)

print(f"Retrieved {len(initial_docs)} documents, narrowed down to top 5 via Reranking.")

Retrieved 20 documents, narrowed down to top 5 via Reranking.


In [15]:
def get_groq_answer(query, context):
    prompt = f"Based on the context, answer: '{query}'. If context is ToC/Index or missing info, reply ONLY 'Insufficient Context'.\n\nContext: {context[:2000]}"
    try:
        comp = client.chat.completions.create(messages=[{"role": "user", "content": prompt}], model="llama-3.1-8b-instant", temperature=0)
        return comp.choices[0].message.content.strip()
    except: return "Error"

target_queries = [
    "What specific city and year was the Prophet born in?",
    "What were the names of the angels who provided shade during the journey to Syria?",
    "Provide specific details about the stay in the Cave of Thawr during Hijrah.",
    "Describe the physical appearance of the seal of prophethood.",
    "List the paternal ancestors of the Prophet starting from Abdullah."
]

print("=== FINAL HUMAN EVALUATION OF RERANKER ===\n")
for i, q in enumerate(target_queries, 1):
    cands = hybrid_retrieval(q, top_k=20)
    raw_ans = get_groq_answer(q, cands[0])

    # Reranking process
    refined_candidates = rerank_results(q, cands)
    ref_ans = get_groq_answer(q, refined_candidates[0])

    print(f"QUERY {i}: {q}")
    print(f"[-] WITHOUT RERANKER (Standard Retrieval):\n{raw_ans}")
    print(f"[+] WITH RERANKER (Cross-Encoder):\n{ref_ans}")

    if "Insufficient" in raw_ans and "Insufficient" not in ref_ans:
        print("VERDICT: ✅ RERANKER CORRECTED FAILURE (Found actual content where search found ToC/Noise)")
    elif raw_ans != ref_ans:
        print("VERDICT: ⚡ RERANKER IMPROVED QUALITY (Surfaced more relevant context)")
    else:
        print("VERDICT: ➖ NO CHANGE (Standard retrieval was already optimal)")
    print("="*80 + "\n")

=== FINAL HUMAN EVALUATION OF RERANKER ===

QUERY 1: What specific city and year was the Prophet born in?
[-] WITHOUT RERANKER (Standard Retrieval):
Insufficient Context
[+] WITH RERANKER (Cross-Encoder):
Makkah, 570 A.D.
VERDICT: ✅ RERANKER CORRECTED FAILURE (Found actual content where search found ToC/Noise)

QUERY 2: What were the names of the angels who provided shade during the journey to Syria?
[-] WITHOUT RERANKER (Standard Retrieval):
Insufficient Context
[+] WITH RERANKER (Cross-Encoder):
Maysarah says: "In the severe heat of the afternoon, I would notice two angels offering shade to Rasulullah ." 

Unfortunately, the names of the angels are not mentioned in the given context.
VERDICT: ✅ RERANKER CORRECTED FAILURE (Found actual content where search found ToC/Noise)

QUERY 3: Provide specific details about the stay in the Cave of Thawr during Hijrah.
[-] WITHOUT RERANKER (Standard Retrieval):
Insufficient Context.
[+] WITH RERANKER (Cross-Encoder):
During their stay in the Cav